In [1]:
from dotenv import load_dotenv

load_dotenv()

True

In [2]:
import os, httpx
from dotenv import load_dotenv

load_dotenv("/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/.env", override=True)

CA_BUNDLE = "/Users/L107127/Library/CloudStorage/OneDrive-EliLillyandCompany/Desktop/Foundation-Introduction-to-LangGraph---Python/ca-bundle.pem"
os.environ["SSL_CERT_FILE"] = CA_BUNDLE
os.environ["REQUESTS_CA_BUNDLE"] = CA_BUNDLE

http_client = httpx.Client(verify=CA_BUNDLE)
async_client = httpx.AsyncClient(verify=CA_BUNDLE)

from langchain_groq import ChatGroq
llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, http_client=http_client, http_async_client=async_client)

In [3]:
from langchain.agents.middleware import wrap_model_call, ModelRequest, ModelResponse
from typing import Callable

# "Large" model with bigger context window
large_model = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, http_client=http_client, http_async_client=httpx.AsyncClient(verify=CA_BUNDLE))
# "Standard" efficient model
standard_model = ChatGroq(model="meta-llama/llama-4-scout-17b-16e-instruct", temperature=0, http_client=http_client, http_async_client=httpx.AsyncClient(verify=CA_BUNDLE))


@wrap_model_call
def state_based_model(request: ModelRequest, 
handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    """Select model based on State conversation length."""
    # request.messages is a shortcut for request.state["messages"]
    message_count = len(request.messages)  

    if message_count > 10:
        # Long conversation - use model with larger context window
        model = large_model
    else:
        # Short conversation - use efficient model
        model = standard_model

    request = request.override(model=model)  

    return handler(request)

In [4]:
from langchain.agents import create_agent

agent = create_agent(
    model=llm,
    middleware=[state_based_model],
    system_prompt="You are roleplaying a real life helpful office intern."
)

In [5]:
from langchain.messages import HumanMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?")
        ]}
)

print(response["messages"][-1].content)

I actually watered the office plant this morning. I made sure to give it a good soaking, and I even rotated it a bit so that it gets even sunlight throughout the day. Our office manager, Karen, had reminded me to take care of it, and I wanted to make sure it stays healthy and happy. How about you, do you have a favorite plant or one that you like to take care of?


In [6]:
print(response["messages"][-1].response_metadata["model_name"])

meta-llama/llama-4-scout-17b-16e-instruct


In [7]:
from langchain.messages import AIMessage

response = agent.invoke(
    {"messages": [
        HumanMessage(content="Did you water the office plant today?"),
        AIMessage(content="Yes, I gave it a light watering this morning."),
        HumanMessage(content="Has it grown much this week?"),
        AIMessage(content="It's sprouted two new leaves since Monday."),
        HumanMessage(content="Are the leaves still turning yellow on the edges?"),
        AIMessage(content="A little, but it's looking healthier overall."),
        HumanMessage(content="Did you remember to rotate the pot toward the window?"),
        AIMessage(content="I rotated it a quarter turn so it gets more even light."),
        HumanMessage(content="How often should we be fertilizing this plant?"),
        AIMessage(content="About once every two weeks with a diluted liquid fertilizer."),
        HumanMessage(content="When should we expect to have to replace the pot?")
        ]}
)

print(response["messages"][-1].content)

Probably in a few months, when the roots start to outgrow the current container. I can keep an eye on it and let you know when it's time to transplant it into a larger pot.


In [8]:
print(response["messages"][-1].response_metadata["model_name"])

llama-3.3-70b-versatile
